<a href="https://colab.research.google.com/github/reen24/MASDFR_20271/blob/entrega-practica1/sol_temp_1_%2C_2%2C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ✅ Práctica 1 — Equipo 6 (ramo: Crédito)
## Matemáticas Actuariales para Seguro de Daños, Fianzas y Reaseguro · UNAM

### Integrantes:
-
-
-
-
-
-


## 1. Carga y análisis de datos

In [1]:
df = pd.read_parquet("/content/cartera.parquet")

NameError: name 'pd' is not defined

In [2]:
%matplotlib inline
import numpy as np #Cargamos las paqueterias que vamos a utilizar
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
np.random.seed(0); plt.rcParams["figure.figsize"] = (8, 3.6)  #Definimos una semilla
# y el tamaño de las gráficas

df = pd.read_parquet("/content/cartera.parquet")  #Cargamos nuestra base de datos
print(df.columns)   #Imprimimos las variables que tenemos

display(df.head(10))  #Mostramos las primeras 10 filas de nuestro dataframe


FileNotFoundError: [Errno 2] No such file or directory: '/content/cartera.parquet'

In [ ]:
print("Pólizas:", len(df), "| Columnas:", len(df.columns))
print("\nVariables utilizadas en la Práctica 1:")
print("  frecuencia : num_siniestros, exposicion")
print("  severidad  : monto_promedio_siniestro (ground-up)")
print("  póliza     : deducible, suma_asegurada, coaseguro")
print("\nLos factores de riesgo no se utilizan en esta práctica; se reservarán para la Práctica 2.")

required = [
    "num_siniestros", "exposicion", "monto_promedio_siniestro",
    "deducible", "suma_asegurada", "coaseguro"
]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Faltan columnas requeridas: {missing}")



con = df[df["num_siniestros"] > 0].copy()

print(f"% de pólizas con siniestro : {(df.num_siniestros > 0).mean():.1%}")
print(f"Exposición total           : {df.exposicion.sum():,.2f} años-póliza")
print(f"Siniestros totales         : {df.num_siniestros.sum():,.0f}")
print(f"Frecuencia simple           : {df.num_siniestros.mean():.4f} siniestros/póliza")
print(f"Tasa por exposición         : {df.num_siniestros.sum()/df.exposicion.sum():.4f} siniestros/año-póliza")
print(f"Severidad ground-up → media : {con.monto_promedio_siniestro.mean():,.2f}")
print(f"Severidad ground-up → mediana: {con.monto_promedio_siniestro.median():,.2f}")
print(f"Máxima severidad observada  : {con.monto_promedio_siniestro.max():,.2f}")

# Medidas simples de asimetría y dispersión.
razon_mm = con.monto_promedio_siniestro.mean() / con.monto_promedio_siniestro.median()
cv = con.monto_promedio_siniestro.std() / con.monto_promedio_siniestro.mean()

print(f"\nMedia / mediana            : {razon_mm:.2f}")
print(f"Coeficiente de variación  : {cv:.2f}")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))

vals, freqs = np.unique(df.num_siniestros, return_counts=True)
ax[0].bar(vals, freqs)
ax[0].set_title("Número de siniestros por póliza")
ax[0].set_xlabel("N")
ax[0].set_ylabel("Pólizas")

ax[1].hist(con.monto_promedio_siniestro, bins=60)
ax[1].set_title("Severidad ground-up")
ax[1].set_xlabel("Monto promedio por siniestro")

plt.tight_layout()
plt.show()

La **media** de la severidad ground-up nos va a estar indicando la pérdida esperada por incumplimiento. Recordemos que:
$$
{\pi} = \text{E(N)} \times \text{E(X)}.
$$
En nuestra cartera, la media es 190,382, lo que indica que, en promedio, un crédito que incumple genera una pérdida no recuperada de ese orden.

La mediana nos va a indicar en que porcentaje se encuentra nuestra media, es decir hacia donde se esta moviendo la cola de nuestra cartera o bien hacia donde se concentran los mayores "deudores".

Un CV alto confirma que la cartera es "heterogénea": conviven créditos pequeños con créditos muy grandes, típico de carteras de crédito con mezcla de deudores.

La forma de la cola se describe mediante los percentiles altos, la asimetría y la curtosis.



## 2. Severidad

La severidad \(X\) representa el monto de un siniestro. En esta práctica trabajamos con `monto_promedio_siniestro`, que corresponde a la severidad **ground-up**, es decir, el monto real del siniestro antes de aplicar las condiciones de la póliza.

Se ajustan cuatro distribuciones candidatas mediante máxima verosimilitud:

- Lognormal
- Gamma
- Weibull
- Pareto

La selección se realiza considerando **AIC y comportamiento de la cola**. Después se distingue la severidad ground-up de la que finalmente paga la aseguradora al aplicar deducible, coaseguro y suma asegurada.

In [ ]:
x = con["monto_promedio_siniestro"].dropna().values

def aic(dist, params, data):
    k = len(params)
    return 2 * k - 2 * np.sum(dist.logpdf(data, *params))

candidatas = {
    "Lognormal": stats.lognorm,
    "Gamma": stats.gamma,
    "Weibull": stats.weibull_min,
    "Pareto": stats.pareto
}

ajustes = {}
filas = []

for nombre, dist in candidatas.items():
    params = dist.fit(x, floc=0)
    ajustes[nombre] = (dist, params)
    filas.append({
        "Distribución": nombre,
        "AIC": aic(dist, params, x)
    })

tabla_sev = (
    pd.DataFrame(filas)
    .sort_values("AIC")
    .reset_index(drop=True)
)

tabla_sev["Delta AIC"] = tabla_sev["AIC"] - tabla_sev["AIC"].min()

print(tabla_sev.to_string(
    index=False,
    formatters={
        "AIC": "{:,.1f}".format,
        "Delta AIC": "{:,.1f}".format
    }
))

mejor_sev = tabla_sev.loc[0, "Distribución"]

print(f"\n→ Mejor AIC: {mejor_sev}")

La función de supervivencia permite estudiar directamente la probabilidad de observar siniestros grandes.

En la gráfica de supervivencia se utiliza una escala log-log para hacer más visible la diferencia entre las colas de las distribuciones. La selección definitiva debe considerar si la curva ajustada sigue razonablemente a la supervivencia empírica en la región de grandes pérdidas.

In [ ]:
p99_emp = np.percentile(x, 99)
grid = np.linspace(x.min(), p99_emp, 400)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))

# Densidad: se muestra hasta el p99 para visualizar mejor el cuerpo.
ax[0].hist(
    x,
    bins=70,
    density=True,
    range=(x.min(), p99_emp)
)

for nombre, (dist, params) in ajustes.items():
    ax[0].plot(
        grid,
        dist.pdf(grid, *params),
        lw=2,
        label=nombre
    )

ax[0].set_title("Densidad ajustada")
ax[0].set_xlabel("Severidad")
ax[0].set_ylabel("Densidad")
ax[0].legend(fontsize=8)

# Supervivencia empírica y supervivencias ajustadas.
xs = np.sort(x)
S_emp = 1 - np.arange(1, len(xs) + 1) / len(xs)

ax[1].plot(xs, S_emp, ".", ms=2, label="Empírica")

for nombre, (dist, params) in ajustes.items():
    ax[1].plot(
        xs,
        dist.sf(xs, *params),
        lw=2,
        label=nombre
    )

ax[1].set_xscale("log")
ax[1].set_yscale("log")
ax[1].set_ylim(1e-4, 1)
ax[1].set_title("Cola: supervivencia en escala log-log")
ax[1].set_xlabel("Severidad")
ax[1].set_ylabel("S(x) = P(X > x)")
ax[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

# Información del modelo seleccionado.
dist_sev, pr_sev = ajustes[mejor_sev]

print(f"E[X] ground-up del modelo {mejor_sev}: "
      f"{dist_sev.mean(*pr_sev):,.2f}")

var99_emp = np.percentile(x, 99)
var99_modelo = dist_sev.ppf(0.99, *pr_sev)

print(f"VaR99 empírico: {var99_emp:,.2f}")
print(f"VaR99 {mejor_sev}: {var99_modelo:,.2f}")

Para nuestra cartera, la **Lognormal** obtiene el menor AIC. Además, en la gráfica de supervivencia presenta un comportamiento consistente con la cola empírica, mientras que Gamma y Weibull caen más rápidamente en la región extrema y Pareto mantiene una cola considerablemente más pesada.

Por ello, la distribución seleccionada para modelar la severidad ground-up es:

$$
\boxed{\text{Lognormal}}
$$



In [ ]:
if mejor_sev == "Lognormal":
    sigma = pr_sev[0]
    loc = pr_sev[1]
    scale = pr_sev[2]
    mu = np.log(scale)

    print("Parametrización de la Lognormal:")
    print(f"mu    = {mu:.6f}")
    print(f"sigma = {sigma:.6f}")
    print(f"loc   = {loc:.6f}")
    print(f"scale = {scale:.2f}")
    print("\nModelo:")
    print(f"ln(X) ~ N({mu:.6f}, {sigma:.6f}²)")
else:
    print("La distribución seleccionada no fue Lognormal.")

### Groud-up vs Transformada

Recordemos que La severidad ground-up representa el daño real. La severidad transformada representa el monto que paga la aseguradora después de aplicar las condiciones de la póliza.

Para cada póliza se consideran:

- \(d\): deducible;
- \(L\): suma asegurada;
- `coas`: proporción del siniestro a cargo del asegurado.

Siguiendo la transformación utilizada en la práctica:

$$
X_{\text{pago}}
=
\min\left\{
\max(X-d,0)(1-\text{coas}),\,L
\right\}.
$$

Así:

- el deducible elimina la parte inferior del pago;
- el coaseguro reduce proporcionalmente el pago;
- la suma asegurada establece el máximo que paga la aseguradora.

Es importante mantener esta distinción porque ajustar la distribución sobre datos ya transformados significa modelar la severidad pagada y no la severidad ground-up.

In [ ]:
# --- Severidad que paga la aseguradora -------------------------------

def transformar(x, ded, coas, suma_asegurada):
    pago_neto = np.maximum(x - ded, 0) * (1 - coas)
    return np.minimum(pago_neto, suma_asegurada)

pagado = transformar(
    con["monto_promedio_siniestro"].values,
    con["deducible"].values,
    con["coaseguro"].values,
    con["suma_asegurada"].values
)

media_ground_up = con["monto_promedio_siniestro"].mean()
media_transformada = pagado.mean()

mediana_ground_up = con["monto_promedio_siniestro"].median()
mediana_transformada = np.median(pagado)

reduccion_media = 1 - media_transformada / media_ground_up

print(f"E[X] ground-up    : {media_ground_up:,.2f}")
print(f"E[X] transformada : {media_transformada:,.2f}")

print(f"\nMediana ground-up    : {mediana_ground_up:,.2f}")
print(f"Mediana transformada : {mediana_transformada:,.2f}")

print(
    f"\nReducción de la severidad media por las condiciones "
    f"de la póliza: {reduccion_media:.2%}"
)

La cartera presenta una severidad asimétrica a la derecha, por lo que se compararon distintas distribuciones de severidad y no se eligió el modelo únicamente de manera arbitraria.

La **Lognormal** fue seleccionada al presentar el menor AIC y un comportamiento de cola consistente con la supervivencia empírica. La comparación de las colas es especialmente importante porque las pérdidas grandes tienen un efecto relevante sobre el riesgo y sobre medidas como los cuantiles.

Finalmente, se distinguió entre la severidad **ground-up** y la severidad **transformada**, aplicando las condiciones contractuales de cada póliza. La comparación de sus medias muestra cuánto se reduce el monto esperado que paga la aseguradora respecto al daño ground-up.

Con esto se obtiene la severidad que servirá como entrada para la etapa posterior de frecuencia y pérdida agregada.

## 3. Frecuencia

La frecuencia representa el número de siniestros que presenta cada póliza. Usaremos la variable `num_siniestros` para la cantidad de siniestros que obtuvo la poliza y `exposicion` el tiempo que estuvo activa.

Para la modelación usaremos las distribuciones **Poisson** y **Binomial Negativa**. Antes de seleccionar un modelo se analizamos la dispersión de los datos y la presencia de ceros. Además, se utilizamos quasi-Poisson como diagnóstico de dispersión.

Dado que las polizas de nuestra cartera tienen diferentes niveles de exposicion, la tasa de frecuencia la calcularemos considerando la exposición total.

Para la expocision usaremos la formula

$$
\hat{\lambda}=
\frac{\sum_i N_i}{\sum_i e_i},
$$

donde \(N_i\) es el número de siniestros de la póliza \(i\) y \(e_i\) es su exposición.

In [ ]:
# ─── Poisson, quasi-Poisson y binomial negativa (con offset) ───────────────

y = df.num_siniestros.values
off = np.log(df.exposicion.values)
Xc = np.ones((len(df), 1))

pois = sm.GLM(
    y, Xc,
    family=sm.families.Poisson(),
    offset=off
).fit()

quasi = sm.GLM(
    y, Xc,
    family=sm.families.Poisson(),
    offset=off
).fit(scale="X2")

nb = sm.NegativeBinomial(
    y, Xc,
    offset=off
).fit(disp=0)

lam_hat = np.exp(pois.params[0])
phi = quasi.scale

print(f"Tasa λ̂ (siniestros por año-póliza) = {lam_hat:.4f}")
print(f"Índice de dispersión (Var/media)   = {y.var()/y.mean():.3f}")
print(f"Factor de dispersión φ (quasi)      = {phi:.3f}   (>1 → sobredispersión)")

print(f"\nAIC Poisson           = {pois.aic:,.0f}")
print(f"AIC Binomial negativa = {nb.aic:,.0f}")

print(
    f"→ Gana: "
    f"{'Binomial negativa' if nb.aic < pois.aic else 'Poisson'}"
)

Nuestro indice de dispersion nos da a entender que existe una leve sobredispersion, de igual forma, nuestro factor de dispersion nos confirma que existe una leve sobredispersion en nuestra cartera.

Dado que buscamos tener un menor AIC para un mejor equilibrio entre nuestro ajuste y complejidad, la Binomial Negativa gana con una diferencia de 26 puntos.

Seleccionamos la Binomial negativa para modelar la frecuencia, por un mejor ajuste frente la sobredispersion presentada.

